# DOFA + lc-col BigEarthNet Phase 1

Colab-first workflow for real Sentinel-2 chips from `lc-col/bigearthnet` with DOFA torch.hub mode. Main path: 64-sample quick check, then 5000-sample Phase 1 run with chunked embeddings. No fake chips are generated.


In [ ]:
# 1. Clone or update this repo
# Edit this if your fork/repo URL differs.
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"
REPO_DIR = "rsfm-fairness-audit"

from pathlib import Path
if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}


In [ ]:
# 2. Install the package
!python -m pip install -e .


In [ ]:
# 3. Install DOFA + Hugging Face/HDF5 dependencies
!python -m pip install -r requirements-dofa.txt


In [ ]:
# 4. Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# Configure DOFA for lc-col 12-band Sentinel-2 + torch.hub mode.
from pathlib import Path
import yaml

DATA_ROOT_64 = "data/bigearthnet_lccol_subset"
DATA_ROOT_5000 = "data/bigearthnet_lccol_subset5000"
OUTPUT_64 = "outputs/dofa_bigearthnet_lccol64"
OUTPUT_5000 = "outputs/dofa_bigearthnet_lccol5000"
CACHE_DIR = "data/_cache/lc_col_bigearthnet"
config_path = Path("configs/models/dofa.yaml")
config = yaml.safe_load(config_path.read_text())
config["band_profile"] = "sentinel2_12_lccol"
config["expected_bands"] = 12
config["repo_path"] = None
config["checkpoint_path"] = None
config["allow_torch_hub_download"] = True
config["device"] = "auto"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())


## Real lc-col BigEarthNet Data

The next cell downloads one real `lc-col/bigearthnet` HDF5 train shard from Hugging Face, inspects its HDF5 keys, and converts the first 64 real Sentinel-2 chips to this project's adapter format. The shard is large, so run this in Colab rather than committing data to Git.


In [ ]:
# Download one real lc-col/bigearthnet HDF5 shard and convert 64 real Sentinel-2 chips
!python scripts/download_bigearthnet_lccol_subset.py \
  --output-dir {DATA_ROOT_64} \
  --cache-dir {CACHE_DIR} \
  --max-samples 64 \
  --seed 42


In [ ]:
# Run preflight checker
!python -m rsfm_fairness_audit.cli check-real \
  --model dofa \
  --dataset bigearthnet \
  --model-config configs/models/dofa.yaml \
  --data-root {DATA_ROOT_64}


In [ ]:
# Run 64-sample real DOFA + lc-col BigEarthNet smoke test
!python -m rsfm_fairness_audit.cli run-real \
  --dataset bigearthnet \
  --dataset-root {DATA_ROOT_64} \
  --model dofa \
  --config configs/models/dofa.yaml \
  --output-dir {OUTPUT_64} \
  --max-samples 64


In [ ]:
# Inspect 64-sample outputs
!find {OUTPUT_64} -maxdepth 3 -type f -print
!sed -n '1,160p' {OUTPUT_64}/report.md

from IPython.display import Image, display
for fig in [f"{OUTPUT_64}/figures/average_vs_worst_group.png", f"{OUTPUT_64}/figures/fairness_map.png", f"{OUTPUT_64}/figures/raw_vs_balanced_gap.png"]:
    if Path(fig).exists():
        display(Image(filename=fig))


## 5000-Sample Phase 1 Run

Run this after the 64-sample quick check succeeds. It reuses `data/_cache/lc_col_bigearthnet` and enables chunked extraction to avoid Colab free-RAM OOM.


In [ ]:
# Convert 5000 real chips and run Phase 1 with chunked embeddings
!python scripts/download_bigearthnet_lccol_subset.py \
  --output-dir {DATA_ROOT_5000} \
  --cache-dir {CACHE_DIR} \
  --max-samples 5000 \
  --seed 42

!python -m rsfm_fairness_audit.cli run-real \
  --dataset bigearthnet \
  --dataset-root {DATA_ROOT_5000} \
  --model dofa \
  --config configs/models/dofa.yaml \
  --output-dir {OUTPUT_5000} \
  --max-samples 5000 \
  --chunk-size 256 \
  --streaming-embeddings true


In [ ]:
# Inspect 5000-sample Phase 1 outputs
!find {OUTPUT_5000} -maxdepth 3 -type f -print
!sed -n '1,160p' {OUTPUT_5000}/report.md

from IPython.display import Image, display
for fig in [f"{OUTPUT_5000}/figures/average_vs_worst_group.png", f"{OUTPUT_5000}/figures/fairness_map.png", f"{OUTPUT_5000}/figures/raw_vs_balanced_gap.png"]:
    if Path(fig).exists():
        display(Image(filename=fig))

!ls -lh {OUTPUT_5000}/tables


In [ ]:
# Zip outputs for download
!zip -r dofa_bigearthnet_phase1_outputs.zip {OUTPUT_64} {OUTPUT_5000}
from google.colab import files
files.download("dofa_bigearthnet_phase1_outputs.zip")
